[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# seq2seq 中的注意力

这个 notebook 改编自 Sean Robertson 的 pytorch 教程 [NLP FROM SCRATCH: TRANSLATION WITH A SEQUENCE TO SEQUENCE NETWORK AND ATTENTION](https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html)，并由 Marc Lelarge 为[深度学习课程](https://dataflowr.github.io/website/) 改编。


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random
import numpy as np
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 数据存放位置：
# 如果在 COLAB 上，可以注释掉下面几行
import os
from pathlib import Path

ROOT_DIR = Path.home()
data_path = os.path.join(ROOT_DIR,'data/')

# 如果在 COLAB 上，取消注释下面几行，可以用下面的命令下载数据：
#!wget https://download.pytorch.org/tutorial/data.zip
#!unzip data.zip
#data_path = './'

In [ ]:
def running_mean(x, N=100):
    cumsum = np.cumsum(np.insert(x, 0, 0)) 
    return (cumsum[N:] - cumsum[:-N]) / float(N)

# 数据预处理

这段代码直接取自 PyTorch 教程，创建法语-英语句子对的语料，以及法语和英语的分词器。


In [ ]:
SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # 统计 SOS 和 EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

# 把 Unicode 字符串转成纯 ASCII，感谢
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# 转小写、去空白、去掉非字母字符

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # 读取文件并按行分割
    lines = open(data_path+'data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # 把每行分割成句子对并归一化
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # 反转句子对，创建 Lang 实例
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPair_test(p):
    return len(p[0].split(' ')) > MAX_LENGTH and \
        len(p[1].split(' ')) > MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def filterPairs_test(pairs):
    return [pair for pair in pairs if filterPair_test(pair)]

def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(random.choice(pairs))

In [ ]:
pairs_train, pairs_val = train_test_split(pairs, test_size=0.2, random_state=42)

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [ ]:
val_pairs = [tensorsFromPair(pv) for pv in pairs_val]

In [ ]:
val_pairs[0]

# Seq2seq

这里我们遵循 PyTorch 教程，实现 [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215v3)。代码里唯一的修改是：编码器在前向传播时接收整个句子（不需要隐状态输入），并输出所有对应的隐状态。这样一来，编码器就不需要写 for 循环。不过，为了简单起见，我们不处理 batch；如果你想处理 batch，请看 [PyTorch 中带序列的批处理](https://dataflowr.github.io/website/modules/11c-batches-with-sequences/)。

我们也在测试集上训练，并在验证集上计算损失（见上面语料的划分）。


In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input):
        embedded = self.embedding(input)
        output, _ = self.gru(embedded, self.initHidden())
        return output

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:
n_iters = 4
training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]

In [ ]:
training_pairs[0][0]

In [ ]:
training_pairs[0][1]

In [ ]:
hidden_size = 256
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)

In [ ]:
one_input = training_pairs[0][0]
out = encoder(one_input)
# 下面是教程里用的代码：
#encoder_hidden = encoder.initHidden()
#for c in one_input:
#    out, encoder_hidden = encoder(c,encoder_hidden)

In [ ]:
out.shape

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        #用 self.softmax = nn.LogSoftmax(dim=1) 会让人误解记号……

    def forward(self, input, hidden, *kwargs):
        output = self.embedding(input).view(1, 1, -1)
        #output = F.relu(output) 为什么对 embedding 取 relu？
        output, hidden = self.gru(output, hidden)
        #output = self.softmax(self.out(output[0]))
        return F.log_softmax(self.out(output[0]),dim=1), hidden, None  # 不必担心最后一个参数

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

In [ ]:
decoder_input = torch.tensor([[SOS_token]], device=device)
decoder_hidden = out[-1,:,:].view(1,1,hidden_size)

In [ ]:
decoder_hidden.shape

In [ ]:
output, hidden, _ = decoder(decoder_input, decoder_hidden)

In [ ]:
def train_onepair(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, teacher_forcing_ratio = 0.5, with_attention = False):
    encoder_hidden = encoder.initHidden()
    
    encoder = encoder.train()
    decoder = decoder.train()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    target_length = target_tensor.size(0)
    loss = 0
    encoder_outputs = encoder(input_tensor)
    decoder_input = torch.tensor([[SOS_token]], device=device)
    
    if with_attention:
        decoder_hidden = decoder.initHidden()
    else:
        decoder_hidden = encoder_outputs[-1,:,:].view(1,1,hidden_size)
    
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # 教师强制：把目标词作为下一个输入
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # 教师强制

    else:
        # 不用教师强制：用自己的预测作为下一个输入
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # 从历史中 detach 作为输入
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [ ]:
def eval_onepair(input_tensor, target_tensor, encoder, decoder, criterion, teacher_forcing_ratio = 0.5, with_attention = False):
    encoder_hidden = encoder.initHidden()

    encoder = encoder.eval()
    decoder = decoder.eval()
    target_length = target_tensor.size(0)
    loss = 0
    encoder_outputs = encoder(input_tensor)
    decoder_input = torch.tensor([[SOS_token]], device=device)
    
    if with_attention:
        decoder_hidden = decoder.initHidden()
    else:
        decoder_hidden = encoder_outputs[-1,:,:].view(1,1,hidden_size)
        
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # 教师强制：把目标词作为下一个输入
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # 教师强制

    else:
        # 不用教师强制：用自己的预测作为下一个输入
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # 从历史中 detach 作为输入
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    return loss.item() / target_length

In [ ]:
def trainIters(encoder, decoder, n_iters, print_every=1000, learning_rate=0.01, teacher_forcing_ratio=0.9, with_attention = False):
    
    plot_losses = []
    plot_losses_val = []
    print_loss_total = 0  # 每隔 print_every 重置

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs_train))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train_onepair(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion, teacher_forcing_ratio, with_attention)
        print_loss_total += loss
        plot_losses.append(loss)

        if iter % print_every == 0:
            loss_val = 0
            for (input_tensor, target_tensor) in val_pairs:
                loss = eval_onepair(input_tensor, target_tensor, encoder,
                     decoder, criterion, teacher_forcing_ratio, with_attention)
                loss_val += loss
            loss_val = loss_val/len(val_pairs)
            plot_losses_val.append(loss_val)
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('(%d %d%%) loss train %.4f and val %.4f' % (iter, iter / n_iters * 100, print_loss_avg, loss_val))
    return plot_losses, plot_losses_val

In [ ]:
learning_rate=0.01
encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()
train_onepair(training_pairs[0][0],training_pairs[0][1],encoder,decoder,encoder_optimizer,decoder_optimizer,criterion)

In [ ]:
hidden_size = 128
n_epochs = 50000
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)
plot_losses,plot_losses_val = trainIters(encoder,decoder,n_epochs,print_every=2500)

In [ ]:
plt.plot(running_mean(plot_losses))
plt.plot([2500*i for i in range(int(n_epochs/2500))], plot_losses_val)

In [ ]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH, with_attention = False):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = encoder(input_tensor)

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        if with_attention:
            decoder_hidden = decoder.initHidden()
        else:
            decoder_hidden = encoder_outputs[-1,:,:].view(1,1,hidden_size)
        
        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden,_ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words

In [ ]:
def evaluateRandomly(encoder, decoder, n=10, with_attention = False):
    for i in range(n):
        pair = random.choice(pairs_val)
        print('>', pair[0])
        print('=', pair[1])
        output_words = evaluate(encoder, decoder, pair[0], with_attention =with_attention)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [ ]:
evaluateRandomly(encoder, decoder)

# 注意力机制

这里我们实现论文 [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473) 中启发而来的注意力机制。下面的代码和原始 PyTorch 教程有显著不同……


In [ ]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.embedding = nn.Embedding(self.output_size, self.hidden_size, padding_idx=0)
        self.attn_w = nn.Linear(2 * self.hidden_size, self.hidden_size)
        self.attn_v = nn.Linear(self.hidden_size, 1)
        self.gru = nn.GRU(self.hidden_size * 2, self.hidden_size)
        self.out = nn.Linear(self.hidden_size * 2, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        # B = 1 批大小
        # encoder_outputs (L,B,H)
        seq_len, _, _ = encoder_outputs.shape
        # hidden (1,B,H)
        hidden_tile = hidden.repeat(seq_len, 1, 1) # (L,B,H)
        # 拼接
        concat = torch.cat((hidden_tile, encoder_outputs), dim=2) # (L,B,2*H)
        attn_weights = F.softmax(self.attn_v(torch.tanh(self.attn_w(concat))), dim=0)
        # attn_weights (L,B,1) torch.sum(attn_weights,dim=0) = 1
        contexts = torch.einsum('jbi,jbk->ibk', attn_weights, encoder_outputs) # (1,B,H)
        # 输入是上一个预测
        embedded = self.embedding(input).view(1,1,-1) # (1,B,H)
        output = torch.cat((embedded, contexts), 2) # (1,B,2*H)
        output, hidden = self.gru(output, hidden)
        # output (1,B,H)
        output = F.log_softmax(self.out(torch.cat((output[0], contexts[0]), 1)),dim=1) # (B,O)
        return output, hidden, attn_weights
        
    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:
hidden_size = 256
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
one_input = training_pairs[0][0]
out = encoder(one_input)

In [ ]:
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

In [ ]:
decoder_input = torch.tensor([[SOS_token]], device=device)
decoder_hidden = decoder.initHidden()
encoder_outputs = out

In [ ]:
decoder_input.shape

In [ ]:
output, hidden, attn_weights = decoder(decoder_input, decoder_hidden, encoder_outputs)

In [ ]:
attn_weights.shape

In [ ]:
torch.sum(attn_weights,dim=0)

In [ ]:
learning_rate=0.01
encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()
train_onepair(training_pairs[0][0],training_pairs[0][1],encoder,decoder,encoder_optimizer,decoder_optimizer,criterion, with_attention = True)

In [ ]:
hidden_size = 256
n_epochs = 50000
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)
plot_losses,plot_losses_val = trainIters(encoder,decoder,n_epochs,print_every=2500,with_attention = True)

In [ ]:
plt.plot(running_mean(plot_losses))
plt.plot([2500*i for i in range(int(n_epochs/2500))], plot_losses_val)

In [ ]:
def evaluate_attention(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = encoder(input_tensor)

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = decoder.initHidden()

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            #print(decoder_attention.shape)
            decoder_attentions[di,:input_length] = decoder_attention[:,0,0].data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1,:input_length]

In [ ]:
francais = "c est un jeune directeur plein de talent ."#"elle a cinq ans de moins que moi ."
output_words, attentions = evaluate_attention(encoder, decoder, francais)

In [ ]:
attentions.shape

In [ ]:
torch.sum(attentions,dim=1)

In [ ]:
plt.matshow(attentions.numpy())

In [ ]:
indexesFromSentence(input_lang, francais)

In [ ]:
tensorFromSentence(input_lang, francais)

In [ ]:
output_words

In [ ]:
evaluateRandomly(encoder, decoder, with_attention =True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
def showAttention(input_sentence, output_words, attentions):
    # 建立带 colorbar 的图
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.numpy(), cmap='bone')
    fig.colorbar(cax)

    # 建立坐标轴
    ax.set_xticklabels([''] + input_sentence.split(' ') +
                       ['<EOS>'],rotation =90)
    ax.set_yticklabels([''] + output_words)

    # 在每个刻度显示标签
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()


def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate_attention(
        encoder, decoder, input_sentence)
    print('input =', input_sentence)
    print('output =', ' '.join(output_words))
    showAttention(input_sentence, output_words, attentions)

In [ ]:
evaluateAndShowAttention("elle a cinq ans de moins que moi .")

In [ ]:
evaluateAndShowAttention("elle est trop petite .")

In [ ]:
evaluateAndShowAttention("je ne crains pas de mourir .")

In [ ]:
evaluateAndShowAttention("c est un jeune directeur plein de talent .")

回到教程，那里编码的注意力机制有点奇怪：注意力权重只由上一个预测 $y_{t-1}$ 和解码器的隐状态 $s_{t-1}$ 计算得到。特别是，注意力权重并不是基于编码器的输出计算的！实际上，无论句子多长，注意力权重的维度总是 `MAX_LENGTH`（而解码器的输出是填充过的）。下面（来自教程）的插图说明了这段代码：

![](https://pytorch.org/tutorials/_images/attention-decoder-network.png)


In [ ]:
class AttnDecoderRNN_tuto(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttnDecoderRNN_tuto, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        attn_weights = F.softmax(
            self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
        
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)

        output = F.relu(output)
        output, hidden = self.gru(output, hidden)

        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:
decoder_tuto = AttnDecoderRNN_tuto(hidden_size, output_lang.n_words).to(device)

In [ ]:
decoder_input = torch.tensor([[SOS_token]], device=device)
decoder_hidden = decoder_tuto.initHidden()
encoder_outputs = torch.randn(MAX_LENGTH, hidden_size).to(device)
output, hidden, attn_weights = decoder_tuto(decoder_input, decoder_hidden, encoder_outputs)

In [ ]:
embedded = decoder_tuto.embedding(decoder_input)
embedded.shape

In [ ]:
hidden = decoder_hidden
hidden.shape

In [ ]:
torch.cat((embedded[0], hidden[0]), 1).shape

In [ ]:
attn_weights = F.softmax(decoder_tuto.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
attn_weights.shape

In [ ]:
attn_weights.unsqueeze(0).shape

In [ ]:
encoder_outputs.unsqueeze(0).shape

In [ ]:
attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

In [ ]:
attn_applied.shape

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)